____
# COMPUTE GRADIENT, INTERPOLATE AND ROTATE SWOT DATA ON COLOC POINTS

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon, coloc_swot, rotate_ggd

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

___________
# CHOOSE

In [2]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v0.nc'
spectral_decomp = True
dt = '12h' #'nearestswath'
if spectral_decomp == True : drifters_sources = 'spectral_decomp_'+ drifters_sources
colocs_sources = f'{dt}_'+drifters_sources

In [3]:
ggd_variables = ['cvl_mean_dynamic_topography_cnes_cls_22',
                 'cvl_mean_sea_surface_cnes_22_hybrid',
                 'cvl_ocean_tide_fes_2022',
                 'cvl_ssha_reference',
                 'duacs_ssha_karin_2_calibrated',
                 'duacs_ssha_karin_2_filtered',]

variables = ['ancillary_surface_classification_flag',
             'cross_track_distance',
             'cvl_distance_to_coast',
             'cvl_swh_model',
             'duacs_editing_flag',
             'duacs_phase_screen',
             'duacs_phase_screen_orbit',
             'duacs_phase_screen_static',
             'duacs_relative_vorticity',
             'duacs_speed_meridional',
             'duacs_speed_meridional_abs',
             'duacs_speed_zonal',
             'duacs_speed_zonal_abs',
             'duacs_strain',
             'duacs_xcal',
             'sig0_karin_2',
             'ssh_karin_2_qual',
             'phi',]

_______
# Data

In [4]:
dfs = browse_swot_250().reset_index()
if spectral_decomp == True : drifters_sources = 'spectral_decomp_'+ drifters_sources
df = pd.read_csv(os.path.join(zarr_dir,'drifters', f'drifterscoloc_'+colocs_sources.replace('.nc', '.csv')), dtype={'drifter_id':str}, parse_dates=['datetime'])

# Create and store coloc SWOT 250m 

In [5]:
method, cutoff = 'naive', ''
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
#rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir,'alti', 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

In [6]:
method, cutoff = 'gauss', 1e3
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir, 'alti', 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


NameError: name 'dfr' is not defined

In [28]:
method, cutoff = 'gauss', 2e3
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir, 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

In [29]:
method, cutoff = 'gauss', 3e3
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir, 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

In [30]:
method, cutoff = 'gauss', 4e3
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir, 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

In [31]:
method, cutoff = 'gauss', 5e3
dfsg = coloc_swot(df, dfs, method_gradient=method, cutoff=cutoff, ggd_variables = ggd_variables, variables=variables)
rotate_ggd(dfsg, ggd_variables)
dfsg.to_csv(os.path.join(zarr_dir, 'swot_250_'+colocs_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh